# Introdução

Projeto desafio conforme a aula de WebConf01 de **Introdução à Robótica do curso de IA Tuma VI** propósta pela Profa. Natássya Barlate.

O robo irá andar pelo cenário e identificar os 2 objetos presentes nele, **Cone Vermelho** e o **Cubo Vermelho**.

## Lógica utilizada

Para este desafio utilizarei a CNN já pré treinada MobileNet. Pois ela é uma CNN na qual já foi treinada e que precisamos apenas fazer alguns ajustes (caso necessáiro) em suas últimas camadas, com isso não precisamos fazer uma CNN do zero.

O motivo de utilizar aa MobileNet em vez de outras como a ResNet foi pelo seu foco bater com o propósito e o cenário. A MobileNet possuí foco em eficiência, velocidade e baixo consumo de bateria, com seu tamanho de modelo extremamente leve, ideal para sistemas embarcados como próprio Robô.

# Etapa 1 - Importações

In [2]:
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalMaxPooling2D, Dropout
from tensorflow.keras.optimizers import Adam

# Etapa 2 - Pré-Processamento

O mobileNetV2 usa:
- 224x224
- preprocessamento próprio

In [3]:
IMG_SIZE = (224,224)
BATCH_SIZE = 32

# Etapa 3 - Data Augmentation

In [4]:
train_datagen = ImageDataGenerator(
    preprocessing_function = tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range = 20,
    zoom_range = 0.2,
    horizontal_flip = True
)

val_datagen = ImageDataGenerator(
    preprocessing_function = tf.keras.applications.mobilenet_v2.preprocess_input
)

# Etapa 4 - Carregar Dataset

In [6]:
train_data = train_datagen.flow_from_directory(
    '../dataset/train',
    target_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    class_mode = 'binary'
)

val_data = val_datagen.flow_from_directory(
    '../dataset/eval',
    target_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    class_mode = 'binary'
)

test_data = val_datagen.flow_from_directory(
    '../dataset/test',
    target_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    class_mode = 'binary',
    shuffle=False
)

Found 316 images belonging to 2 classes.


Found 66 images belonging to 2 classes.
Found 87 images belonging to 2 classes.


# Etapa 5 - Carregar MobileNetV2

In [7]:
base_model = MobileNetV2(
    weights = 'imagenet',
    include_top = False,
    input_shape = (224,224,3)
)

E0000 00:00:1779298527.596311   66197 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1779298527.596646   78009 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1779298527.674030   66197 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


# Etapa 6 - Congelar Rede

In [8]:
# impede que o conhecimento já aprendido não seja perdido
base_model.train = False

# Etapa 7 - Adicionar camadas finais

In [9]:
x = base_model.output
x = GlobalMaxPooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(
    inputs = base_model.input,
    outputs = predictions
)

# Etapa 8 - Compilar modelo

In [10]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Etapa 9 - Treinar

In [11]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10


I0000 00:00:1779299661.297647   66197 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
W0000 00:00:1779299695.821460   78023 cpu_allocator_impl.cc:82] Allocation of 51380224 exceeds 10% of free system memory.
W0000 00:00:1779299695.881398   78023 cpu_allocator_impl.cc:82] Allocation of 51380224 exceeds 10% of free system memory.
W0000 00:00:1779299695.908979   78023 cpu_allocator_impl.cc:82] Allocation of 51380224 exceeds 10% of free system memory.
W0000 00:00:1779299695.937974   78024 cpu_allocator_impl.cc:82] Allocation of 51380224 exceeds 10% of free system memory.
W0000 00:00:1779299695.971596   78024 cpu_allocator_impl.cc:82] Allocation of 51380224 exceeds 10% of free system memory.


10/10 ━━━━━━━━━━━━━━━━━━━━ 89s 5s/step - accuracy: 0.6551 - loss: 1.5969 - val_accuracy: 0.7424 - val_loss: 0.6628
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 50s 5s/step - accuracy: 0.8987 - loss: 0.3138 - val_accuracy: 0.9242 - val_loss: 0.1606
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 49s 5s/step - accuracy: 0.9494 - loss: 0.1448 - val_accuracy: 0.9091 - val_loss: 0.1507
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 50s 5s/step - accuracy: 0.9905 - loss: 0.0410 - val_accuracy: 0.9848 - val_loss: 0.0615
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 49s 5s/step - accuracy: 0.9937 - loss: 0.0196 - val_accuracy: 1.0000 - val_loss: 0.0337
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 49s 5s/step - accuracy: 0.9968 - loss: 0.0061 - val_accuracy: 0.9697 - val_loss: 0.0482
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 50s 5s/step - accuracy: 0.9968 - loss: 0.0044 - val_accuracy: 0.9848 - val_loss: 0.0430
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 50s 5s/step - accuracy: 0.9968 - loss: 0.0065 - val_accuracy: 0.9848 - val_loss: 0.0506
Epo

# Etapa 10 - Avaliar

In [12]:
loss, acc = model.evaluate(test_data)

print('Accuracy', acc)

3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 837ms/step - accuracy: 1.0000 - loss: 0.0166
Accuracy 1.0


# Etapa 11 - Salvar Modelo

In [14]:
model.save('../models/mobilenet_model.h5')